# Worldwide Earthquake Events API - Gold Layer Processing

In [1]:
from pyspark.sql.functions import when, col, udf
from pyspark.sql.types import StringType
# ensure the below library is installed on your fabric environment
import reverse_geocoder as rg
from datetime import date, timedelta

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 5, Finished, Available, Finished, False)

In [3]:
# Setting stard_date and end_date
from datetime import date, timedelta
start_date = date.today() - timedelta(7)
end_date = date.today() - timedelta(1)

df = spark.read.table("earthquake_events_silver").filter(col('time') > start_date)

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 7, Finished, Available, Finished, False)

In [4]:
def get_country_code(lat, lon):
    """
    Retrieve the country code for a given latitude and longitude.

    Parameters:
    lat (float or str): Latitude of the location.
    lon (float or str): Longitude of the location.

    Returns:
    str: Country code of the location, retrieved using the reverse geocoding API.

    Example:
    >>> get_country_details(48.8588443, 2.2943506)
    'FR'
    """
    coordinates = (float(lat), float(lon))
    return rg.search(coordinates)[0].get('cc')

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 8, Finished, Available, Finished, False)

In [5]:
# registering the udfs so they can be used on spark dataframes
get_country_code_udf = udf(get_country_code, StringType())

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 9, Finished, Available, Finished, False)

In [6]:
# adding country_code and city attributes
df_with_location = \
                df.\
                    withColumn("country_code", get_country_code_udf(col("latitude"), col("longitude")))

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 10, Finished, Available, Finished, False)

In [7]:
# adding significance classification
df_with_location_sig_class = \
                            df_with_location.\
                                withColumn('sig_class', 
                                            when(col("sig") < 100, "Low").\
                                            when((col("sig") >= 100) & (col("sig") < 500), "Moderate").\
                                            otherwise("High")
                                            )

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 11, Finished, Available, Finished, False)

In [8]:
# appending the data to the gold table
df_with_location_sig_class.write.mode('append').saveAsTable('earthquake_events_gold')

StatementMeta(, 1245c8d3-782d-4b6d-b096-a1c2c2a43ef2, 12, Finished, Available, Finished, False)